# Unit 1 solutions: build the first scoreboard

These reference solutions use the generated data, keep the test split sealed, and include executable checks.

In [ ]:
import json
from math import comb

import bookrec
import numpy as np
import pandas as pd

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")

## Exercise 1 — Inspect the real catalog schema

In [ ]:
book = catalog[min(catalog)]
item_id = book.item_id
title = book.title
author_id = int(book.fields["author_id"])
genres = book.fields["genres"].split(";")
year = int(book.fields["year"])

assert (item_id, title, author_id, genres, year) == (
    0,
    "Book 00000",
    175,
    ["biography", "poetry", "adventure"],
    1983,
)
item_id, title, author_id, genres, year

## Exercise 2 — Make an analysis table

In [ ]:
catalog_rows = [
    {
        "item_id": book.item_id,
        "title": book.title,
        "author_id": int(book.fields["author_id"]),
        "genres": book.fields["genres"].split(";"),
        "year": int(book.fields["year"]),
    }
    for book in catalog.values()
]
catalog_df = pd.DataFrame(catalog_rows)
first_five = catalog_df.head(5)

assert first_five.shape == (5, 5)
assert list(first_five["item_id"]) == [0, 1, 2, 3, 4]
first_five

## Exercise 3 — Search by text and attributes

In [ ]:
matches = bookrec.search_catalog(catalog, text="science", year=2020)
first_match_ids = [book.item_id for book in matches[:5]]
second_matches = bookrec.search_catalog(catalog, text="mystery", genre="mystery")
second_match_ids = [book.item_id for book in second_matches[:5]]

assert first_match_ids == [377, 489, 903, 1052, 1542]
assert second_match_ids == [2, 7, 21, 40, 45]
first_match_ids, second_match_ids

## Exercise 4 — Retrieve, then rank without repeats

In [ ]:
candidates = [
    bookrec.Candidate(10, 0.99, "practice"),
    bookrec.Candidate(20, 0.95, "practice"),
    bookrec.Candidate(40, 0.80, "practice"),
    bookrec.Candidate(30, 0.80, "practice"),
    bookrec.Candidate(50, 0.70, "practice"),
]
seen = {10, 20}
ranked = bookrec.rank(candidates, n=3, exclude=seen)
ranked_ids = [candidate.item_id for candidate in ranked]

assert ranked_ids == [30, 40, 50]
assert all(candidate.item_id not in seen for candidate in ranked)
ranked

## Exercise 5 — Protect the temporal validation split

In [ ]:
interactions = pd.read_csv(data_dir / "interactions.csv.gz")
reader_id = 0
reader_rows = interactions[interactions["reader_id"] == reader_id]
seen = set(
    reader_rows[
        (reader_rows["split"] == "train") & (reader_rows["label"] == 1)
    ]["item_id"]
)
relevant = set(
    reader_rows[
        (reader_rows["split"] == "val") & (reader_rows["label"] == 1)
    ]["item_id"]
)
leakage_explanation = (
    "Sorting by time keeps later behavior out of training, so the model cannot learn from the future."
)

assert seen == {309, 419, 424, 907, 941, 949, 1223, 1549, 1747, 1757, 1771, 1996}
assert relevant == {400}
seen, relevant, leakage_explanation

## Exercise 6 — Compute the two top-k metrics

In [ ]:
recommended = np.array([8, 13, 21, 34])
relevant = np.array([13, 55])
top_three_matches = np.isin(recommended[:3], relevant)
hit_from_scratch = float(top_three_matches.any())
recall_from_scratch = float(top_three_matches.sum() / len(relevant))
hit_from_library = bookrec.hit_rate_at_k(recommended, relevant, k=3)
recall_from_library = bookrec.recall_at_k(recommended, relevant, k=3)

assert (hit_from_scratch, recall_from_scratch) == (1.0, 0.5)
assert (hit_from_library, recall_from_library) == (
    hit_from_scratch,
    recall_from_scratch,
)
hit_from_library, recall_from_library

## Exercise 7 — Challenge: derive the random floor

A uniform `k`-subset is the prefix of a uniformly random permutation. With one relevant item, that item occupies each of the `N` positions with probability `1/N`, so the first `k` positions give `k/N`. More generally, `P(hit) = 1 - C(N-r, k) / C(N, k)`, while `E[recall@k] = k/N`.

In [ ]:
N = 80
k = 8
expected_hit_rate = k / N
condition = "0 <= k <= N"
why_unseen_only = (
    "A recommender must not return previously read books, so its candidate population is the unseen catalog."
)

assert expected_hit_rate == 0.1
expected_hit_rate, condition, why_unseen_only

## Exercise 8 — Challenge: analytic versus empirical

In [ ]:
small_catalog = range(12)
small_seen = {0, 1}
one_relevant = {2}
small_N = 10
small_k = 2
trial_hits = []
for seed in range(2000):
    random_path = bookrec.RandomRetrievalPath(small_catalog, seed=seed)
    candidates = random_path.retrieve(99, {"seen": small_seen}, k=small_k)
    recommendations = bookrec.rank(candidates, n=small_k, exclude=small_seen)
    trial_hits.append(bookrec.hit_rate_at_k(recommendations, one_relevant, k=small_k))
empirical_hit_rate = float(np.mean(trial_hits))
analytic_hit_rate = small_k / small_N

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])
demo_k = 50
scoreboard = bookrec.run_validation_scoreboard(
    bookrec.RandomRetrievalPath(catalog, seed=2026),
    data_dir / "interactions.csv.gz",
    catalog_ids=catalog,
    k=demo_k,
    cold_readers=cold_readers,
)
interactions = pd.read_csv(data_dir / "interactions.csv.gz")
analytic_hits = []
val_positive = interactions[(interactions["split"] == "val") & (interactions["label"] == 1)]
for reader_id in sorted(set(val_positive["reader_id"]) - cold_readers):
    reader_rows = interactions[interactions["reader_id"] == reader_id]
    seen = set(reader_rows[(reader_rows["split"] == "train") & (reader_rows["label"] == 1)]["item_id"])
    relevant = set(reader_rows[(reader_rows["split"] == "val") & (reader_rows["label"] == 1)]["item_id"]) - seen
    if relevant:
        unseen_count = len(catalog) - len(seen)
        miss_probability = comb(unseen_count - len(relevant), demo_k) / comb(unseen_count, demo_k)
        analytic_hits.append(1 - miss_probability)
analytic_expected_hit_rate = float(np.mean(analytic_hits))
comparison = (
    "The empirical estimate is close to k/N; the aggregate validation score averages "
    "per-reader hit probabilities based on each reader's unseen and relevant sets, so it need not equal one k/N."
)

assert analytic_hit_rate == 0.2
assert abs(empirical_hit_rate - analytic_hit_rate) < 0.03
assert scoreboard.readers > 0
assert scoreboard.hit_rate_at_k > 0
{
    "small_empirical_hit_rate": empirical_hit_rate,
    "small_analytic_hit_rate": analytic_hit_rate,
    "scoreboard_empirical_hit_rate_at_50": scoreboard.hit_rate_at_k,
    "scoreboard_analytic_expected_hit_rate_at_50": analytic_expected_hit_rate,
    "scoreboard_empirical_recall_at_50": scoreboard.recall_at_k,
    "comparison": comparison,
}